# 04 — Interactive Inference

Load a fine-tuned model (SFT or RL) and interactively ask it math questions
with streaming token output.

**Requirements:** Google Colab with GPU (T4 minimum).

## 1. Setup

In [ ]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/YOUR_USERNAME/math-rl-tuning.git
%cd math-rl-tuning

!pip install -e . --quiet
!pip install bitsandbytes --quiet

In [ ]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, mount_google_drive

cfg = load_config()
setup_hf_token()
mount_google_drive()

## 2. Load Model

In [ ]:
from math_rl_tuning.model import load_adapter

# Choose which adapter to load:
ADAPTER_PATH = cfg.paths.sft_output_dir       # SFT model
# ADAPTER_PATH = cfg.paths.grpo_output_dir     # RL model
# ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"  # From Drive

model, tokenizer = load_adapter(cfg, adapter_path=ADAPTER_PATH)
print(f"Model loaded on: {model.device}")

## 3. Interactive Q&A (Streaming)

In [ ]:
from math_rl_tuning.inference import generate_stream, extract_final_answer

question = "Given a sequence {a_n} with a_1 = 1 and a_{n+1} = 3*S_n, find a_6."

print(f"Question: {question}\n")
print("=" * 50)
response = generate_stream(
    question, model, tokenizer,
    max_new_tokens=cfg.inference.max_new_tokens,
    temperature=cfg.inference.temperature,
)
print("=" * 50)

answer = extract_final_answer(response)
print(f"\nExtracted answer: {answer}")

In [ ]:
# Try more questions
questions = [
    "What is the sum of the first 100 positive integers?",
    "Solve for x: x^2 - 5x + 6 = 0",
    "A store sells apples for $1.50 each. If you buy 3 apples and pay with a $10 bill, how much change do you get?",
    "Find the derivative of f(x) = x^3 + 2x^2 - 5x + 1",
]

for q in questions:
    print(f"\nQ: {q}")
    print("-" * 50)
    response = generate_stream(q, model, tokenizer, max_new_tokens=512)
    answer = extract_final_answer(response)
    print(f"\nExtracted: {answer}")
    print("=" * 60)

## 4. Non-Streaming Generation (for scripting)

In [ ]:
from math_rl_tuning.inference import generate_from_config

response = generate_from_config(
    "How many prime numbers are less than 20?",
    model, tokenizer, cfg,
    stream=False,
)

print(response)

## 5. Batch Inference on Test Set

In [ ]:
from math_rl_tuning.data import prepare_test_data
from math_rl_tuning.inference import generate, extract_final_answer
from math_rl_tuning.utils import extract_boxed
from tqdm import tqdm

test_ds = prepare_test_data(cfg)

# Run on first 10 examples
for i in range(min(10, len(test_ds))):
    ex = test_ds[i]
    question = ex["problem"]
    gold = extract_boxed(ex["solution"])

    response = generate(question, model, tokenizer, max_new_tokens=512, do_sample=False)
    pred = extract_final_answer(response)

    match = "OK" if pred and gold and pred.strip() == gold.strip() else "MISS"
    print(f"[{match}] Gold={gold}  Pred={pred}  | {question[:60]}...")

## 6. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory

del model
clean_memory()